In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib


# Loading the dataset and Data Engineering
df = pd.read_csv('flood_final_dataset.csv')
df = df.drop(columns=["lat","lon","timestamp","state"]) 

#encoding landslides
susceptibility_order = {"Low": 0, "Moderate": 1, "High": 2}
df["landslide_susceptibility"] = df["landslide_susceptibility"].map(susceptibility_order)

# encode target, same ordinal logic
risk_order = {"Low": 0, "Medium": 1, "High": 2, "Severe": 3}
df["label_flood_risk"] = df["label_flood_risk"].map(risk_order)

X = df.drop(columns=["label_flood_risk", "region", "water_level_m", "danger_level_m"])
y = df["label_flood_risk"]
grps = df["region"]

gkf = GroupKFold(n_splits=5)
accs = []  # NEW: track accuracy per fold

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=grps)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=4,
        class_weight='balanced',
    )
    model.fit(X_train, y_train,categorical_feature=["landslide_susceptibility","month"])
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)  # NEW
    accs.append(acc)                      # NEW

    print(f"--- Fold {fold} --- accuracy: {acc:.4f}")  # UPDATED
    print(classification_report(y_test, preds))

# NEW: your reportable score for judges
mean_acc = sum(accs) / len(accs)
print(f"\nMean accuracy across folds (report this number): {mean_acc:.4f}")

# Train final model on ALL data (for deployment/demo use)
final_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=4,
    class_weight='balanced',
)
final_model.fit(X, y, categorical_feature=["landslide_susceptibility","month"])

# NEW: save the final model so Streamlit can load it later
joblib.dump(final_model, "flood_model.pkl")
print("Saved final model to flood_model.pkl")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000927 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1571
[LightGBM] [Info] Number of data points in the train set: 115200, number of used features: 10
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
--- Fold 0 --- accuracy: 0.8279
              precision    recall  f1-score   support

           0       0.87      0.87      0.87     21605
           1       0.78      0.76      0.77     13482
           2       0.63      0.91      0.75       716
           3       0.93      0.97      0.95       197

    accuracy                           0.83     36000
   macro avg       0.80      0.88      0.83     36000
weighted avg      